# Final Validation Set Model Evaluation Pipeline

This notebook loads the frozen validation dataset split, queries the MLflow SQLite backend 
to fetch the highest-performing model pipeline artifact, and computes generalization metrics.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import mlflow
import pandas as pd
import numpy as np

from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

/Users/hector.vargas/repos/ml_hands_on_project/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Environment Configurations & Data Ingestion

In [2]:
# Define core path anchors relative to notebook location
ROOT_DIR = os.path.abspath("../")
DB_PATH = os.path.join(ROOT_DIR, "mlflow.db")

VAL_FEATURES_PATH = "../data/processed/raw_features/X_val.csv"
VAL_TARGET_PATH = "../data/processed/target/y_val.csv"

In [3]:
# Load validation partitions
X_val = pd.read_csv(VAL_FEATURES_PATH)
y_val = pd.read_csv(VAL_TARGET_PATH).squeeze()

print(f"Validation features shape: {X_val.shape}")
print(f"Validation target distribution:\n{y_val.value_counts(normalize=True)}")

Validation features shape: (1600, 17)
Validation target distribution:
Exited
0    0.79625
1    0.20375
Name: proportion, dtype: float64


## 2. Query MLflow Registry for the Absolute Best Run

In [ ]:
# Point tracking client to the centralized SQLite server
mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")

In [5]:
# Search across all non-deleted experiments to find the top performing run
all_experiments = mlflow.search_experiments()
experiment_ids = [exp.experiment_id for exp in all_experiments]

print(f"Scanning {len(experiment_ids)} experiments for optimal run records...")

Scanning 4 experiments for optimal run records...


In [6]:
# Search for the best single run using available AUC metrics as sort keys
runs_df = mlflow.search_runs(
    experiment_ids=experiment_ids,
    order_by=["metrics.pr_auc DESC", "metrics.test_auc DESC", "metrics.best_auc DESC"],
    max_results=1
)

if runs_df.empty:
    raise RuntimeError("No active model runs found in the MLflow tracking database.")

best_run = runs_df.iloc[0]

In [15]:
# Extract experiment details safely
parent_exp = mlflow.get_experiment(best_run["experiment_id"])

print("=" * 60)
print(f"BEST PIPELINE FOUND FOUND")
print("=" * 60)
print(f"Experiment Name : {parent_exp.name}")
print(f"Model Flavor    : {best_run.get('tags.mlflow.runName', 'Unnamed Model')}")
print(f"MLflow Run ID   : {best_run.run_id}")
print(f"Training Score  : {best_run.get('metrics.pr_auc', best_run.get('metrics.best_auc', 0.0)):.4f} (PR-AUC)")
print("=" * 60)

BEST PIPELINE FOUND FOUND
Experiment Name : customer-churn-simple
Model Flavor    : logistic_regression
MLflow Run ID   : 95fba8e756974d98b8a19cc28170ed36
Training Score  : 0.9991 (PR-AUC)


## 3. Load the Serialized Production Pipeline Model Object

In [8]:
# Construct direct model path URI using the run ID
model_uri = f"runs:/{best_run.run_id}/best_model"

try:
    # Try loading the optimized optuna model path first
    best_pipeline = mlflow.sklearn.load_model(model_uri)
except Exception:
    # Fallback path if utilizing standard baseline run wrappers
    model_uri = f"runs:/{best_run.run_id}/model"
    best_pipeline = mlflow.sklearn.load_model(model_uri)

print(f"Successfully unpacked model architecture state from: {model_uri}")

Successfully unpacked model architecture state from: runs:/95fba8e756974d98b8a19cc28170ed36/model


## 4. Execute Generalization Inference on Validation Cohort

In [9]:
# Generate hard predictions and raw probability vectors
y_pred = best_pipeline.predict(X_val)
y_proba = best_pipeline.predict_proba(X_val)[:, 1]

## 5. Compute Generalization Performance Metrics

In [10]:
# Calculate precision, recall, and dual area under the curves
precision = precision_score(y_val, y_pred, zero_division=0)
recall = recall_score(y_val, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_val, y_proba)
pr_auc = average_precision_score(y_val, y_proba)

In [11]:
# Create a clean display frame
metrics_summary = pd.DataFrame({
    "Metric Name": ["Precision", "Recall (Sensitivity)", "ROC-AUC (Ranking)", "PR-AUC (Average Precision)"],
    "Validation Score": [precision, recall, roc_auc, pr_auc]
})

In [12]:
# Display clean tabular layout
print("\n### Final Validation Assessment Profile ###")
display(metrics_summary.style.format({"Validation Score": "{:.4f}"}))


### Final Validation Assessment Profile ###


,Metric Name,Validation Score
0,Precision,0.9969
1,Recall (Sensitivity),0.9939
2,ROC-AUC (Ranking),0.9994
3,PR-AUC (Average Precision),0.9981


### 5.1 Granular Classification Report Matrix

In [13]:
print("\n" + classification_report(y_val, y_pred, zero_division=0))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1274
           1       1.00      0.99      1.00       326

    accuracy                           1.00      1600
   macro avg       1.00      1.00      1.00      1600
weighted avg       1.00      1.00      1.00      1600



### 5.2 Confusion Matrix Count Outputs

In [14]:
cm = confusion_matrix(y_val, y_pred)
cm_df = pd.DataFrame(
    cm, 
    index=['Actual Retained (0)', 'Actual Churned (1)'], 
    columns=['Predicted Retained (0)', 'Predicted Churned (1)']
)
display(cm_df)

,Predicted Retained (0),Predicted Churned (1)
Actual Retained (0),1273,1
Actual Churned (1),2,324
